# 06-P4. paper-aligned feature attribution

이 노트북은 reconstruction error 기반 대체 attribution을 계산해 `main_abnormal_features`와 설명 필드를 만든다.

현재 구현 제약:
- 논문에서 사용한 ARCANA는 현재 프로젝트 환경에 없다.
- 따라서 baseline attribution은 `scaled reconstruction squared error`를 feature importance proxy로 사용한다.


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml을 찾을 수 없습니다.')


def direction_label(value: float) -> str:
    if value > 0:
        return 'higher_than_normal'
    if value < 0:
        return 'lower_than_normal'
    return 'neutral'


def prettify_feature_name(column_name: str) -> str:
    return column_name.replace('__', ' / ').replace('_', ' ')


def is_explanation_candidate(feature_name: str) -> bool:
    blocked_exact = {
        'row_count',
        'missing_count',
        'missing_rate',
        'max_timestamp_gap_minutes',
        'extreme_change_count',
        'has_dhw',
        'has_buffer_tank',
    }
    blocked_suffixes = (
        '__missing_count',
        '__missing_rate',
    )
    if feature_name in blocked_exact:
        return False
    if feature_name.endswith(blocked_suffixes):
        return False
    return True


PROJECT_ROOT = find_project_root(Path.cwd())
WINDOW_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_windows' / 'ml_window_dataset.csv'
FEATURE_COLUMNS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_features' / 'feature_columns.csv'
IMPUTATION_VALUES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_features' / 'imputation_values.csv'
EVENT_TIMELINE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned' / 'event_detection_timeline.csv'
MODEL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned' / 'models' / 'paper_aligned_autoencoder_model.joblib'
SCALER_PATH = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned' / 'models' / 'paper_aligned_autoencoder_scaler.joblib'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned'
ATTRIBUTION_OUTPUT = OUTPUT_DIR / 'feature_attribution_summary.csv'
MAIN_FEATURE_OUTPUT = OUTPUT_DIR / 'main_abnormal_features.csv'
METADATA_OUTPUT = OUTPUT_DIR / 'feature_attribution_metadata.json'

feature_columns_df = pd.read_csv(FEATURE_COLUMNS_PATH)
feature_columns = feature_columns_df.loc[feature_columns_df['selected_for_baseline'] == True, 'column_name'].tolist()
imputation_df = pd.read_csv(IMPUTATION_VALUES_PATH)
imputation_map = dict(zip(imputation_df['column_name'], imputation_df['imputation_value']))

windows_df = pd.read_csv(WINDOW_PATH, parse_dates=['window_start', 'window_end'])
timeline_df = pd.read_csv(EVENT_TIMELINE_PATH, parse_dates=['window_start', 'window_end', 'event_start', 'event_end', 'report_date'])

autoencoder = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

merge_keys = ['manufacturer', 'substation_id', 'window_start', 'window_end', 'label']
working_df = timeline_df.merge(
    windows_df[merge_keys + feature_columns],
    on=merge_keys,
    how='left',
    validate='one_to_one',
)

X = working_df[feature_columns].copy()
for column in feature_columns:
    X[column] = X[column].fillna(imputation_map[column])
X = X.astype(float)

X_scaled = scaler.transform(X)
X_reconstructed = autoencoder.predict(X_scaled)
squared_error = (X_scaled - X_reconstructed) ** 2
signed_residual = X_scaled - X_reconstructed
row_error_sum = squared_error.sum(axis=1, keepdims=True)
row_error_sum[row_error_sum == 0] = 1.0
normalized_contribution = squared_error / row_error_sum

contribution_columns = [f'{column}__contribution' for column in feature_columns]
residual_columns = [f'{column}__signed_residual' for column in feature_columns]
contribution_df = pd.DataFrame(normalized_contribution, columns=contribution_columns)
residual_df = pd.DataFrame(signed_residual, columns=residual_columns)
working_df = pd.concat([working_df.reset_index(drop=True), contribution_df, residual_df], axis=1)

event_rows = []
main_rows = []
for event_key, event_group in working_df.groupby(['manufacturer', 'event_type', 'event_split', 'event_id'], sort=False):
    manufacturer, event_type, event_split, event_id = event_key
    detected = bool(event_group['criticality_crossed'].astype(bool).any())
    focus_mask = event_group['criticality_crossed'].astype(bool)
    focus_rule = 'criticality_crossed_windows'
    if focus_mask.sum() == 0:
        focus_rule = 'max_anomaly_score_window'
        focus_index = event_group['anomaly_score'].astype(float).idxmax()
        focus_group = event_group.loc[[focus_index]].copy()
    else:
        focus_group = event_group.loc[focus_mask].copy()

    contribution_means = focus_group[contribution_columns].mean(axis=0)
    residual_means = focus_group[residual_columns].mean(axis=0)

    feature_records = []
    for feature_name in feature_columns:
        contribution_value = float(contribution_means[f'{feature_name}__contribution'])
        residual_value = float(residual_means[f'{feature_name}__signed_residual'])
        feature_records.append(
            {
                'feature_name': feature_name,
                'pretty_feature_name': prettify_feature_name(feature_name),
                'contribution': contribution_value,
                'signed_residual_mean': residual_value,
                'direction': direction_label(residual_value),
            }
        )

    feature_rank_df = pd.DataFrame(feature_records)
    feature_rank_df = feature_rank_df[feature_rank_df['feature_name'].map(is_explanation_candidate)].copy()
    feature_rank_df = feature_rank_df.sort_values(['contribution', 'pretty_feature_name'], ascending=[False, True]).reset_index(drop=True)
    feature_rank_df['rank'] = np.arange(1, len(feature_rank_df) + 1)
    top_features = feature_rank_df.head(5).copy()

    for _, feature_row in top_features.iterrows():
        event_rows.append(
            {
                'manufacturer': manufacturer,
                'event_type': event_type,
                'event_split': event_split,
                'event_id': int(event_id),
                'substation_id': int(event_group['substation_id'].iloc[0]),
                'configuration_type': event_group['configuration_type'].iloc[0],
                'fault_label': event_group['fault_label'].iloc[0],
                'detected': detected,
                'focus_rule': focus_rule,
                'focus_window_count': int(len(focus_group)),
                'rank': int(feature_row['rank']),
                'feature_name': feature_row['feature_name'],
                'pretty_feature_name': feature_row['pretty_feature_name'],
                'contribution': float(feature_row['contribution']),
                'signed_residual_mean': float(feature_row['signed_residual_mean']),
                'direction': feature_row['direction'],
            }
        )

    top3 = top_features.head(3).copy()
    main_feature_tokens = [
        f"{row.pretty_feature_name} ({row.direction}, {row.contribution:.3f})"
        for row in top3.itertuples(index=False)
    ]
    explanation = '; '.join(main_feature_tokens)
    main_rows.append(
        {
            'manufacturer': manufacturer,
            'event_type': event_type,
            'event_split': event_split,
            'event_id': int(event_id),
            'substation_id': int(event_group['substation_id'].iloc[0]),
            'configuration_type': event_group['configuration_type'].iloc[0],
            'fault_label': event_group['fault_label'].iloc[0],
            'detected': detected,
            'focus_rule': focus_rule,
            'focus_window_count': int(len(focus_group)),
            'main_abnormal_features': ' | '.join(main_feature_tokens),
            'feature_explanation': explanation,
            'top_feature_1': top3.iloc[0]['feature_name'],
            'top_feature_1_direction': top3.iloc[0]['direction'],
            'top_feature_1_contribution': float(top3.iloc[0]['contribution']),
            'top_feature_2': top3.iloc[1]['feature_name'] if len(top3) > 1 else None,
            'top_feature_2_direction': top3.iloc[1]['direction'] if len(top3) > 1 else None,
            'top_feature_2_contribution': float(top3.iloc[1]['contribution']) if len(top3) > 1 else np.nan,
            'top_feature_3': top3.iloc[2]['feature_name'] if len(top3) > 2 else None,
            'top_feature_3_direction': top3.iloc[2]['direction'] if len(top3) > 2 else None,
            'top_feature_3_contribution': float(top3.iloc[2]['contribution']) if len(top3) > 2 else np.nan,
        }
    )

feature_attribution_summary_df = pd.DataFrame(event_rows)
main_abnormal_features_df = pd.DataFrame(main_rows)

feature_attribution_summary_df.to_csv(ATTRIBUTION_OUTPUT, index=False, encoding='utf-8-sig')
main_abnormal_features_df.to_csv(MAIN_FEATURE_OUTPUT, index=False, encoding='utf-8-sig')

metadata = {
    'attribution_method': 'normalized_scaled_reconstruction_squared_error',
    'focus_rule_primary': 'criticality_crossed_windows',
    'focus_rule_fallback': 'max_anomaly_score_window',
    'top_k_saved': 5,
    'main_feature_top_k': 3,
    'limitation': 'ARCANA replacement baseline; not equivalent to the paper method',
}
METADATA_OUTPUT.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

display(feature_attribution_summary_df.head(15))
display(main_abnormal_features_df.head(10))
print(f'saved: {ATTRIBUTION_OUTPUT}')
print(f'saved: {MAIN_FEATURE_OUTPUT}')
print(f'saved: {METADATA_OUTPUT}')


,manufacturer,event_type,event_split,event_id,substation_id,configuration_type,fault_label,detected,focus_rule,focus_window_count,rank,feature_name,pretty_feature_name,contribution,signed_residual_mean,direction
0,manufacturer 1,fault_event,holdout,1,10,SH + DHW,Motorised control valve (primary side): Actuat...,True,criticality_crossed_windows,3,1,dhw_supply_temperature_gap__mean,dhw supply temperature gap / mean,0.620193,-70.244244,lower_than_normal
1,manufacturer 1,fault_event,holdout,1,10,SH + DHW,Motorised control valve (primary side): Actuat...,True,criticality_crossed_windows,3,2,dhw_supply_temperature_gap__last,dhw supply temperature gap / last,0.091453,-26.952718,lower_than_normal
2,manufacturer 1,fault_event,holdout,1,10,SH + DHW,Motorised control valve (primary side): Actuat...,True,criticality_crossed_windows,3,3,s_dhw_supply_temperature__mean,s dhw supply temperature / mean,0.059197,-21.701158,lower_than_normal
3,manufacturer 1,fault_event,holdout,1,10,SH + DHW,Motorised control valve (primary side): Actuat...,True,criticality_crossed_windows,3,4,s_dhw_supply_temperature__last,s dhw supply temperature / last,0.029747,-15.372834,lower_than_normal
4,manufacturer 1,fault_event,holdout,1,10,SH + DHW,Motorised control valve (primary side): Actuat...,True,criticality_crossed_windows,3,5,s_dhw_supply_temperature__first,s dhw supply temperature / first,0.027952,-14.890491,lower_than_normal
5,manufacturer 1,fault_event,holdout,5,11,SH + DHW,Failure of the heating circuit pump,True,criticality_crossed_windows,1,1,s_dhw_lower_storage_temperature__min,s dhw lower storage temperature / min,0.003005,2.159068,higher_than_normal
6,manufacturer 1,fault_event,holdout,5,11,SH + DHW,Failure of the heating circuit pump,True,criticality_crossed_windows,1,2,p_net_supply_temperature__first,p net supply temperature / first,0.002556,1.991251,higher_than_normal
7,manufacturer 1,fault_event,holdout,5,11,SH + DHW,Failure of the heating circuit pump,True,criticality_crossed_windows,1,3,s_dhw_supply_temperature__delta,s dhw supply temperature / delta,0.001712,-1.629484,lower_than_normal
8,manufacturer 1,fault_event,holdout,5,11,SH + DHW,Failure of the heating circuit pump,True,criticality_crossed_windows,1,4,network_temperature_gap__max_abs,network temperature gap / max abs,0.001335,1.439152,higher_than_normal
9,manufacturer 1,fault_event,holdout,5,11,SH + DHW,Failure of the heating circuit pump,True,criticality_crossed_windows,1,5,s_hc1_supply_temperature__std,s hc1 supply temperature / std,0.001286,-1.412585,lower_than_normal


,manufacturer,event_type,event_split,event_id,substation_id,configuration_type,fault_label,detected,focus_rule,focus_window_count,main_abnormal_features,feature_explanation,top_feature_1,top_feature_1_direction,top_feature_1_contribution,top_feature_2,top_feature_2_direction,top_feature_2_contribution,top_feature_3,top_feature_3_direction,top_feature_3_contribution
0,manufacturer 1,fault_event,holdout,1,10,SH + DHW,Motorised control valve (primary side): Actuat...,True,criticality_crossed_windows,3,dhw supply temperature gap / mean (lower_than_...,dhw supply temperature gap / mean (lower_than_...,dhw_supply_temperature_gap__mean,lower_than_normal,0.620193,dhw_supply_temperature_gap__last,lower_than_normal,0.091453,s_dhw_supply_temperature__mean,lower_than_normal,0.059197
1,manufacturer 1,fault_event,holdout,5,11,SH + DHW,Failure of the heating circuit pump,True,criticality_crossed_windows,1,s dhw lower storage temperature / min (higher_...,s dhw lower storage temperature / min (higher_...,s_dhw_lower_storage_temperature__min,higher_than_normal,0.003005,p_net_supply_temperature__first,higher_than_normal,0.002556,s_dhw_supply_temperature__delta,lower_than_normal,0.001712
2,manufacturer 1,fault_event,holdout,3,12,SH + DHW,Control unit: Incorrect parameterisation,False,max_anomaly_score_window,1,s hc1 supply temperature / delta (lower_than_n...,s hc1 supply temperature / delta (lower_than_n...,s_hc1_supply_temperature__delta,lower_than_normal,0.084953,p_hc1_return_temperature__delta,lower_than_normal,0.081143,s_dhw_supply_temperature_setpoint__std,lower_than_normal,0.062405
3,manufacturer 1,fault_event,holdout,53,13,SH + DHW,Shut-off valve closed,True,criticality_crossed_windows,1,s dhw supply temperature setpoint / last (high...,s dhw supply temperature setpoint / last (high...,s_dhw_supply_temperature_setpoint__last,higher_than_normal,0.012909,s_dhw_supply_temperature_setpoint__mean,higher_than_normal,0.010335,s_dhw_supply_temperature_setpoint__min,higher_than_normal,0.010049
4,manufacturer 1,fault_event,holdout,69,26,SH + DHW with sub-circuits,unknown,True,criticality_crossed_windows,1,dhw supply temperature gap / mean (lower_than_...,dhw supply temperature gap / mean (lower_than_...,dhw_supply_temperature_gap__mean,lower_than_normal,0.309736,p_net_supply_temperature__std,higher_than_normal,0.206499,p_net_supply_temperature__delta,higher_than_normal,0.075634
5,manufacturer 2,fault_event,holdout,5,8,SH,Leakage,False,max_anomaly_score_window,1,p hc1 return temperature / max (higher_than_no...,p hc1 return temperature / max (higher_than_no...,p_hc1_return_temperature__max,higher_than_normal,0.002031,p_net_supply_temperature__first,higher_than_normal,0.001930,p_hc1_return_temperature__first,higher_than_normal,0.001463
6,manufacturer 2,fault_event,holdout,1,24,SH + DHW,Shut-off valve closed,False,max_anomaly_score_window,1,p net supply temperature / std (higher_than_no...,p net supply temperature / std (higher_than_no...,p_net_supply_temperature__std,higher_than_normal,0.403484,p_net_return_temperature__std,higher_than_normal,0.067659,s_dhw_supply_temperature__min,lower_than_normal,0.057597
7,manufacturer 2,fault_event,holdout,4,39,SH with buffer tank,Failure of the thermal energy meter,False,max_anomaly_score_window,1,p net supply temperature / std (higher_than_no...,p net supply temperature / std (higher_than_no...,p_net_supply_temperature__std,higher_than_normal,0.183086,p_net_supply_temperature__delta,lower_than_normal,0.178038,s_dhw_upper_storage_temperature__mean,higher_than_normal,0.046929
8,manufacturer 2,fault_event,holdout,3,45,SH,Motorised control valve (primary side): Incorr...,False,max_anomaly_score_window,1,"outdoor temperature / std (higher_than_normal,...","outdoor temperature / std (higher_than_normal,...",outdoor_temperature__std,higher_than_normal,0.265046,p_net_supply_temperature__std,higher_than_normal,0.143068,p_net_meter_heat_power__std,higher_than_normal,0.059726
9,manufacturer 2,fault_event,holdout,2,53,SH 

saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\feature_attribution_summary.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\main_abnormal_features.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\feature_attribution_metadata.json


## 다음 단계

- `main_abnormal_features.csv`를 06-P5에서 Agent 계약 스키마에 직접 연결한다.
- `feature_explanation`은 운영자 설명과 작업지시서 초안의 근거 필드로 사용한다.
